In [58]:
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
import torch
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import LabelEncoder


In [59]:
df = pd.read_csv('https://raw.githubusercontent.com/gscdit/Breast-Cancer-Detection/refs/heads/master/data.csv')
df.head()

,id,diagnosis,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave points_mean,...,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave points_worst,symmetry_worst,fractal_dimension_worst,Unnamed: 32
0,842302,M,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,...,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890,NaN
1,842517,M,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,...,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902,NaN
2,84300903,M,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,...,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758,NaN
3,84348301,M,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,...,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300,NaN
4,84358402,M,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,...,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678,NaN


In [60]:
df.drop(columns=['id','Unnamed: 32'],inplace=True)

In [61]:
X_train, X_test, y_train, y_test = train_test_split(df.iloc[:,1:], df.iloc[:,0],test_size=0.2)

In [62]:
scaler=StandardScaler()
X_train=scaler.fit_transform(X_train)
X_test=scaler.transform(X_test)

In [63]:
encoder=LabelEncoder()
y_train=encoder.fit_transform(y_train)
y_test=encoder.transform(y_test)

In [64]:
X_train_tensor=torch.from_numpy(X_train.astype(np.float32))
X_test_tensor=torch.from_numpy(X_test.astype(np.float32))
y_train_tensor=torch.from_numpy(y_train.astype(np.float32))
y_test_tensor=torch.from_numpy(y_test.astype(np.float32))

In [65]:
class MY_NN(nn.Module):
  def __init__(self,num_features):
    super().__init__()
    self.linear=nn.Linear(num_features,1)
    self.sigmoid=nn.Sigmoid()

  def forward(self,features):
    out=self.linear(features)
    out=self.sigmoid(out)
    return out


In [66]:
learning_rate=0.432
epochs=30


In [67]:
loss_function=nn.BCELoss()

In [68]:
#training pipeline
#model
model=MY_NN(X_train_tensor.shape[1])
#print(model.weights)
#optimizer
optimizer=torch.optim.SGD(model.parameters(),lr=learning_rate)   #model.parameters() retrieves an iterator over all trainable parameters(weights and biases) in a model
#loop
for epoch in range(epochs):
  #forward pass
  y_pred=model(X_train_tensor)
  #print(y_pred)

  #calcualte the loss
  loss=loss_function(y_pred,y_train_tensor.view(-1,1))
  #print(loss.item())
  #zero the gradients
  optimizer.zero_grad()
  #backprop
  loss.backward()
  #update the weights
  optimizer.step()
  #to print loss in each epoch
  print(f'Epoch: {epoch+1} , loss: {loss.item()}')

Epoch: 1 , loss: 0.8558829426765442
Epoch: 2 , loss: 0.25698426365852356
Epoch: 3 , loss: 0.21912506222724915
Epoch: 4 , loss: 0.1953010857105255
Epoch: 5 , loss: 0.17850425839424133
Epoch: 6 , loss: 0.16584019362926483
Epoch: 7 , loss: 0.1558542251586914
Epoch: 8 , loss: 0.1477227360010147
Epoch: 9 , loss: 0.14093919098377228
Epoch: 10 , loss: 0.1351723074913025
Epoch: 11 , loss: 0.13019484281539917
Epoch: 12 , loss: 0.12584486603736877
Epoch: 13 , loss: 0.12200336903333664
Epoch: 14 , loss: 0.11858054250478745
Epoch: 15 , loss: 0.11550726741552353
Epoch: 16 , loss: 0.11272921413183212
Epoch: 17 , loss: 0.11020311713218689
Epoch: 18 , loss: 0.10789390653371811
Epoch: 19 , loss: 0.10577300935983658
Epoch: 20 , loss: 0.1038166806101799
Epoch: 21 , loss: 0.10200519114732742
Epoch: 22 , loss: 0.10032184422016144
Epoch: 23 , loss: 0.09875251352787018
Epoch: 24 , loss: 0.09728509932756424
Epoch: 25 , loss: 0.0959092453122139
Epoch: 26 , loss: 0.09461592882871628
Epoch: 27 , loss: 0.09339731

In [70]:
#model_evaluation
with torch.no_grad():
  y_pred=model.forward(X_test_tensor)
  y_pred=(y_pred>0.5).float()
  accuracy=(y_pred==y_test_tensor).float().mean()
print(f'Accuracy:{accuracy.item()}')

Accuracy:0.5
